<a href="https://colab.research.google.com/github/Benxperia/MRes/blob/main/Lake_inflow_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lake Inflow Analysis

Per-lake inflow channel analysis for Tanana Valley Basin lakes.
Works entirely on GeoTIFF outputs from tvb_hydro_multimodal_rs.py.

Goals:
  1. MAP    — identify and locate inflow channel entry points per lake
  2. PROXY  — estimate relative inflow magnitude (discharge proxy) per channel
  3. CHANGE — track inflow activity and lake water extent change over time

Inputs required (from GEE exports):
  - Lake polygons shapefile
  - MERIT Hydro:  elevation, flow direction (D8), upstream area
  - ArcticDEM:    elevation, slope, TPI
  - Sentinel-2:   MNDWI freshet and summer composites (per year)
  - Sentinel-1:   VV_dB, inundation_frequency freshet composites (per year)
  - JRC:          water occurrence, seasonality, annual water frequency

Outputs (saved to OUTPUT_DIR):
  - GeoPackage of inflow points per lake (with attributes)
  - GeoPackage of upstream catchments per lake
  - CSV time series of inflow activity and lake water area per lake per year
  - PNG summary plots per lake
  - Master CSV summarising all lakes

# Set up

In [ ]:
!pip install geopandas rasterio numpy pandas matplotlib scipy scikit-image pysheds shapely tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

Mounted at /content/drive/


In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.mask
import rasterio.features
import rasterio.transform
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.crs import CRS
from scipy import ndimage
from shapely.geometry import Point, LineString, MultiPoint, shape
from shapely.ops import unary_union
from tqdm import tqdm
import shutil

# Config

In [ ]:
# Root folder where all GEE exports are saved (your Google Drive folder)
DATA_DIR = "/content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/"

# Sub-folders matching the GEE export folder names
SAR_DIR      = os.path.join(DATA_DIR, "Alaska_TRV_SAR")
S2_DIR       = os.path.join(DATA_DIR, "Alaska_TRV_S2")
TERRAIN_DIR  = os.path.join(DATA_DIR, "Alaska_TRV_Terrain")
WATER_DIR    = os.path.join(DATA_DIR, "Alaska_TRV_SurfaceWater")
# LANDSAT_DIR and MODIS_DIR removed — not used in modern snapshot analysis

# Your lake polygons shapefile
LAKE_FILES = {
    "lake_1": "/content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Chis_Outline/Chis_lake.shp",
    "lake_2": "/content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/LL_Outline/LL_lake.shp"
            }
# Column in the shapefile to use as the lake identifier
# Change to match whatever field name your shapefile uses
LAKE_ID_FIELD = "Lake_name"

# Output directory — all results saved here
OUTPUT_DIR = "/content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "vectors"), exist_ok=True)

# Analysis parameters
BUFFER_DISTANCE_M    = 500    # metres to buffer around each lake when searching
                               # for inflow channels — increase for large catchments
UPSTREAM_AREA_THRESH = 0.5    # km² — minimum upstream area to count as a stream
                               # lower = more channels detected, more false positives
MNDWI_WATER_THRESH   = 0.0    # MNDWI > this = water (0.0 is standard; raise to
                               # reduce false positives in wetland margins)
SAR_WATER_THRESH_DB  = -15.0  # VV dB below which pixel is treated as water
INUND_FREQ_THRESH    = 0.25   # fraction of SAR scenes flagged as inundated
                               # for a channel pixel to be considered active
SLOPE_CHANNEL_THRESH = 0.5    # degrees — minimum slope for a channel pixel
                               # (excludes flat lake surface from channel detection)

# Years available in your downloaded data
# Reduced to match the updated GEE export script (summer only, 2022–2024)
S2_YEARS  = [2022, 2023, 2024]
SAR_YEARS = [2022, 2023, 2024]
# JRC: summary mosaic only — no annual files, so this list is unused
JRC_YEARS = []

print("Configuration loaded.")
print(f"  Data root : {DATA_DIR}")
print(f"  Lakes shp : {LAKE_FILES}")
print(f"  Output    : {OUTPUT_DIR}")

Configuration loaded.
  Data root : /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/
  Lakes shp : {'lake_1': '/content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Chis_Outline/Chis_lake.shp', 'lake_2': '/content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/LL_Outline/LL_lake.shp'}
  Output    : /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/


# Utility functions

In [ ]:
def load_raster_clipped(filepath, geometry_gdf, buffer_m=0):
    """
    Load a raster clipped to the bounding box + optional buffer of a GeoDataFrame.
    Returns (data array, transform, crs, nodata).
    Data is always returned in the raster's native CRS.
    """
    if not os.path.exists(filepath):
        return None, None, None, None

    with rasterio.open(filepath) as src:
        # Reproject geometry to raster CRS for clipping
        geom_reproj = geometry_gdf.to_crs(src.crs)
        if buffer_m > 0:
            geom_reproj = geom_reproj.copy()
            geom_reproj['geometry'] = geom_reproj.geometry.buffer(buffer_m)

        shapes = [geom.__geo_interface__ for geom in geom_reproj.geometry]
        try:
            data, transform = rasterio.mask.mask(src, shapes, crop=True, nodata=src.nodata)
            return data, transform, src.crs, src.nodata
        except Exception as e:
            print(f"    Warning: could not clip {os.path.basename(filepath)}: {e}")
            return None, None, None, None


def reproject_match(src_data, src_transform, src_crs,
                    dst_transform, dst_crs, dst_shape, nodata=-9999):
    """Reproject a raster array to match a target grid."""
    dst_data = np.full((src_data.shape[0], dst_shape[0], dst_shape[1]),
                       nodata, dtype=np.float32)
    reproject(
        source=src_data.astype(np.float32),
        destination=dst_data,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=nodata,
        dst_nodata=nodata
    )
    return dst_data


def pixels_to_points(mask_2d, transform, crs, min_distance_m=50):
    """
    Convert a binary mask to point geometries, one per connected region.
    min_distance_m controls minimum separation between returned points.
    """
    labeled, n_features = ndimage.label(mask_2d)
    points = []
    for i in range(1, n_features + 1):
        region = (labeled == i)
        # Use centroid of each connected region
        rows, cols = np.where(region)
        row_c = int(np.median(rows))
        col_c = int(np.median(cols))
        x, y = rasterio.transform.xy(transform, row_c, col_c)
        points.append(Point(x, y))
    return gpd.GeoDataFrame(geometry=points, crs=crs)


def find_file(directory, pattern):
    """Find a file matching a glob pattern. Returns first match or None."""
    matches = glob.glob(os.path.join(directory, pattern))
    return matches[0] if matches else None


def pixel_area_m2(transform):
    """Return pixel area in m² from an affine transform (assumes projected CRS)."""
    return abs(transform.a * transform.e)

# Stage 1 - Flow network & catchment delineation

In [ ]:
def extract_stream_network(lake_gdf, lake_id, buffer_m=BUFFER_DISTANCE_M):
    print(f"\n  [STAGE 1] Flow network — Lake {lake_id}")

    merit_path = find_file(TERRAIN_DIR, "*MERIT_Hydro*")
    if merit_path is None:
        print("    ERROR: MERIT Hydro file not found.")
        return None, None, None, {}

    # Load MERIT clipped tightly to lake + buffer only
    # This is the key fix — clip FIRST, then analyse the small clipped array
    data, transform, crs, nodata = load_raster_clipped(
        merit_path, lake_gdf, buffer_m=buffer_m
    )
    if data is None:
        return None, None, None, {}

    elev   = data[0].astype(float)
    flow_d = data[1].astype(float)
    upa    = data[2].astype(float)
    width  = data[4].astype(float)

    if nodata is not None:
        elev[elev == nodata]     = np.nan
        flow_d[flow_d == nodata] = np.nan
        upa[upa == nodata]       = np.nan

    stream_mask = (upa >= UPSTREAM_AREA_THRESH) & np.isfinite(upa)
    print(f"    Stream pixels in buffer zone: {stream_mask.sum()}")

    if not stream_mask.any():
        print(f"    No stream pixels found — try lowering UPSTREAM_AREA_THRESH")
        return None, None, None, {}

    # ---- Vectorised inflow detection (replaces slow pixel loop) ----
    # Rasterise the lake boundary directly to a mask, then find
    # stream pixels that overlap it — all in NumPy, no Python loop

    lake_reproj  = lake_gdf.to_crs(crs)
    lake_poly    = lake_reproj.geometry.iloc[0]
    lake_poly_buf = lake_poly.buffer(abs(transform.a) * 2)  # 2-pixel boundary buffer

    # Rasterise lake interior and buffered boundary to arrays
    lake_interior_mask = rasterio.features.rasterize(
        [(lake_poly, 1)],
        out_shape=upa.shape,
        transform=transform,
        fill=0, dtype=np.uint8
    ).astype(bool)

    lake_boundary_mask = rasterio.features.rasterize(
        [(lake_poly_buf, 1)],
        out_shape=upa.shape,
        transform=transform,
        fill=0, dtype=np.uint8
    ).astype(bool)

    # Boundary ring = buffered polygon minus interior
    boundary_ring = lake_boundary_mask & ~lake_interior_mask

    # Inflow pixels = stream pixels that fall on the boundary ring
    inflow_mask = stream_mask & boundary_ring

    if not inflow_mask.any():
        print("    No inflows at boundary — widening to full buffer zone...")
        inflow_mask = stream_mask & ~lake_interior_mask

    print(f"    Inflow candidate pixels: {inflow_mask.sum()}")

    # Convert inflow pixels to points — still using connected-region centroids
    # but now only on the small inflow_mask, not 2M pixels
    inflow_candidates = []
    labeled, n = ndimage.label(inflow_mask)
    for i in range(1, n + 1):
        region = (labeled == i)
        rows_r, cols_r = np.where(region)
        r_c = int(np.median(rows_r))
        c_c = int(np.median(cols_r))
        x, y = rasterio.transform.xy(transform, r_c, c_c)
        inflow_candidates.append({
            'geometry':      Point(x, y),
            'upstream_km2':  float(upa[r_c, c_c])    if np.isfinite(upa[r_c, c_c])    else np.nan,
            'river_width_m': float(width[r_c, c_c])  if np.isfinite(width[r_c, c_c])  else np.nan,
            'elevation_m':   float(elev[r_c, c_c])   if np.isfinite(elev[r_c, c_c])   else np.nan,
            'lake_id':       lake_id
        })

    inflow_gdf = gpd.GeoDataFrame(inflow_candidates, crs=crs) if inflow_candidates else None

    if inflow_gdf is not None and len(inflow_gdf) > 0:
        inflow_gdf = inflow_gdf.sort_values('upstream_km2', ascending=False).reset_index(drop=True)
        inflow_gdf['inflow_rank'] = inflow_gdf.index + 1
        print(f"    Inflow candidates: {len(inflow_gdf)}")
        print(f"    Largest upstream area: {inflow_gdf.upstream_km2.max():.2f} km²")

    # ---- Catchment polygon (vectorised) ----
    catch_poly = None
    if inflow_gdf is not None and len(inflow_gdf) > 0:
        max_upa   = inflow_gdf.upstream_km2.max()
        catch_mask = (upa >= UPSTREAM_AREA_THRESH) & (upa <= max_upa) & np.isfinite(upa)
        rows_c, cols_c = np.where(catch_mask)
        if len(rows_c) > 0:
            xs, ys = rasterio.transform.xy(transform, rows_c.tolist(), cols_c.tolist())
            catch_poly = MultiPoint(list(zip(xs, ys))).convex_hull

    upa_stats = {}
    if inflow_gdf is not None:
        upa_stats = {
            'n_inflows':     len(inflow_gdf),
            'max_upa_km2':   float(inflow_gdf.upstream_km2.max()),
            'total_upa_km2': float(inflow_gdf.upstream_km2.sum()),
        }

    # Stream pixels as GeoDataFrame (vectorised — xs/ys arrays not a loop)
    rows_s, cols_s = np.where(stream_mask)
    xs_s, ys_s = rasterio.transform.xy(transform, rows_s.tolist(), cols_s.tolist())
    streams_gdf = gpd.GeoDataFrame(
        geometry=[Point(x, y) for x, y in zip(xs_s, ys_s)], crs=crs
    )

    return streams_gdf, inflow_gdf, catch_poly, upa_stats

# Stage 2 - Water surface detection & inflow configuration

In [ ]:
def detect_water_at_inflows(lake_gdf, inflow_gdf, lake_id,
                            year, season='summer'):
    """
    For a given lake and year, check whether water is detected at each
    inflow candidate point using Sentinel-2 MNDWI and Sentinel-1 VV/inundation.
    Season is 'summer' only — freshet composites no longer downloaded.

    Returns a dict of per-inflow detection results for that year/season.
    """
    if inflow_gdf is None or len(inflow_gdf) == 0:
        return {}

    results = {}

    # ---- Sentinel-2 MNDWI ----
    s2_pattern = f"*S2*{year}*{season}*"
    s2_path    = find_file(S2_DIR, s2_pattern)

    mndwi_vals = {}
    if s2_path:
        data, transform, crs, nodata = load_raster_clipped(
            s2_path, lake_gdf, buffer_m=BUFFER_DISTANCE_M
        )
        if data is not None:
            # Find MNDWI band — it's index 7 in our S2 export (after 6 spectral bands)
            # Band order: Blue,Green,Red,RE1,RE2,RE3,NIR,NIR_narrow,SWIR1,SWIR2,NDVI,NDWI,NDSI,AWEI,AWEInsh
            mndwi_band = data[11] if data.shape[0] > 11 else data[-1]
            mndwi_band = mndwi_band.astype(float)
            if nodata: mndwi_band[mndwi_band == nodata] = np.nan

            inflow_reproj = inflow_gdf.to_crs(crs)
            for idx, row in inflow_reproj.iterrows():
                pt  = row.geometry
                r_i, c_i = rasterio.transform.rowcol(transform, pt.x, pt.y)
                r_i = max(0, min(r_i, mndwi_band.shape[0] - 1))
                c_i = max(0, min(c_i, mndwi_band.shape[1] - 1))
                val = mndwi_band[r_i, c_i]
                mndwi_vals[idx] = float(val) if np.isfinite(val) else np.nan

    # ---- Sentinel-1 SAR ----
    sar_pattern = f"*S1_SAR*{year}*{season}*"
    sar_path    = find_file(SAR_DIR, sar_pattern)

    vv_vals     = {}
    inund_vals  = {}
    if sar_path:
        data, transform, crs, nodata = load_raster_clipped(
            sar_path, lake_gdf, buffer_m=BUFFER_DISTANCE_M
        )
        if data is not None:
            # Band order: VV_dB, VH_dB, SAR_water_mask, VV_VH_ratio_dB, RVI, inundation_frequency
            vv_band    = data[0].astype(float)
            inund_band = data[5].astype(float) if data.shape[0] > 5 else None
            if nodata:
                vv_band[vv_band == nodata] = np.nan
                if inund_band is not None:
                    inund_band[inund_band == nodata] = np.nan

            inflow_reproj = inflow_gdf.to_crs(crs)
            for idx, row in inflow_reproj.iterrows():
                pt  = row.geometry
                r_i, c_i = rasterio.transform.rowcol(transform, pt.x, pt.y)
                r_i = max(0, min(r_i, vv_band.shape[0] - 1))
                c_i = max(0, min(c_i, vv_band.shape[1] - 1))
                vv_vals[idx]    = float(vv_band[r_i, c_i]) if np.isfinite(vv_band[r_i, c_i]) else np.nan
                if inund_band is not None:
                    inund_vals[idx] = float(inund_band[r_i, c_i]) if np.isfinite(inund_band[r_i, c_i]) else np.nan

    # ---- Combine into per-inflow result ----
    for idx in inflow_gdf.index:
        mndwi  = mndwi_vals.get(idx, np.nan)
        vv     = vv_vals.get(idx, np.nan)
        inund  = inund_vals.get(idx, np.nan)

        # Confidence scoring:
        #   +1 point each for S2 water, SAR water, SAR inundation frequency
        #   Maximum score = 3 (all three agree = high confidence inflow)
        score = 0
        s2_water  = (not np.isnan(mndwi)) and (mndwi > MNDWI_WATER_THRESH)
        sar_water = (not np.isnan(vv))    and (vv < SAR_WATER_THRESH_DB)
        sar_inund = (not np.isnan(inund)) and (inund > INUND_FREQ_THRESH)
        if s2_water:  score += 1
        if sar_water: score += 1
        if sar_inund: score += 1

        results[idx] = {
            'year':          year,
            'season':        season,
            'mndwi':         mndwi,
            'vv_db':         vv,
            'inund_freq':    inund,
            's2_water':      s2_water,
            'sar_water':     sar_water,
            'sar_inund':     sar_inund,
            'confidence':    score,   # 0=no evidence, 1=weak, 2=moderate, 3=strong
            'active_inflow': score >= 1
        }

    return results


def measure_lake_water_area(lake_gdf, lake_id, year):
    """
    Measure open water area within the lake polygon for a given year.
    Uses JRC annual water frequency, with S2 MNDWI as a fallback.

    Returns dict with water area stats.
    """
    stats = {'year': year, 'lake_id': lake_id}

    # ---- JRC summary mosaic (single static file, not annual) ----
    # Provides long-term water occurrence context (1984–2021)
    # Only loaded once — same file used for all years
    jrc_path = find_file(WATER_DIR, "*JRC_SurfaceWater_summary*")
    if jrc_path:
        data, transform, crs, nodata = load_raster_clipped(jrc_path, lake_gdf)
        if data is not None:
            # Band order: water_occurrence_pct, water_change_abs,
            #             water_seasonality_months, ever_water_mask
            occurrence  = data[0].astype(float)
            seasonality = data[2].astype(float)
            ever_water  = data[3].astype(float)
            if nodata:
                occurrence[occurrence == nodata]   = np.nan
                seasonality[seasonality == nodata] = np.nan
                ever_water[ever_water == nodata]   = np.nan
            px_area = pixel_area_m2(transform)
            stats.update({
                'jrc_mean_occurrence_pct': float(np.nanmean(occurrence)),
                'jrc_ever_water_km2':      float(np.nansum(ever_water > 0) * px_area / 1e6),
                'jrc_mean_seasonality_months': float(np.nanmean(seasonality)),
            })

    # ---- S2 MNDWI water area (summer only) ----
    s2_path = find_file(S2_DIR, f"*S2*{year}*summer*")
    if s2_path:
        data, transform, crs, nodata = load_raster_clipped(s2_path, lake_gdf)
        if data is not None:
            mndwi = data[11].astype(float) if data.shape[0] > 11 else data[-1].astype(float)
            if nodata: mndwi[mndwi == nodata] = np.nan
            px_area  = pixel_area_m2(transform)
            s2_water = np.nansum(mndwi > MNDWI_WATER_THRESH) * px_area / 1e6
            stats['s2_summer_water_km2'] = s2_water

    return stats

# Stage 3 - Time series analysis

In [ ]:
def build_inflow_timeseries(lake_gdf, inflow_gdf, lake_id):
    """
    For each inflow point, build a year-by-year time series of:
      - Whether the inflow was active (confidence >= 1)
      - MNDWI value at the inflow point
      - VV backscatter at the inflow point
      - Inundation frequency at the inflow point

    Returns a DataFrame with one row per inflow point per year.
    """
    records = []

    years = sorted(set(S2_YEARS) | set(SAR_YEARS))
    for year in tqdm(years, desc=f"    Inflow detection lake {lake_id}"):
        result = detect_water_at_inflows(
            lake_gdf, inflow_gdf, lake_id, year, season='summer'
        )
        for inflow_idx, vals in result.items():
            row = {
                'lake_id':       lake_id,
                'inflow_idx':    inflow_idx,
                'upstream_km2':  inflow_gdf.loc[inflow_idx, 'upstream_km2'],
                'inflow_rank':   inflow_gdf.loc[inflow_idx, 'inflow_rank'],
            }
            row.update(vals)
            records.append(row)

    return pd.DataFrame(records)


def build_lake_area_timeseries(lake_gdf, lake_id):
    """
    Build year-by-year time series of lake open water area.
    Returns a DataFrame with one row per year.
    """
    records = []
    # Loop over S2 years only (2022–2024) — JRC annual stack not downloaded,
    # JRC summary mosaic is loaded once inside measure_lake_water_area
    for year in tqdm(S2_YEARS, desc=f"    Lake water area {lake_id}"):
        stats = measure_lake_water_area(lake_gdf, lake_id, year)
        records.append(stats)
    return pd.DataFrame(records)

# Stage 4 - Visualisation

In [ ]:
def plot_lake_summary(lake_id, inflow_gdf, inflow_ts, lake_area_ts,
                      upa_stats, lake_gdf):
    """
    Four-panel summary figure per lake:
      Panel A: Inflow point map with upstream area as size
      Panel B: Lake water area time series (JRC + S2)
      Panel C: Inflow activity heatmap (inflow × year)
      Panel D: MNDWI at top 3 inflows over time
    """
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(f"Lake {lake_id} — Inflow Channel Analysis",
                 fontsize=16, fontweight='bold', y=0.98)

    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    # ---- Panel A: Inflow map ----
    if inflow_gdf is not None and len(inflow_gdf) > 0:
        lake_reproj = lake_gdf.to_crs(inflow_gdf.crs)
        # Added aspect='equal' to force equal scaling for projected CRS
        lake_reproj.boundary.plot(ax=ax1, color='steelblue', linewidth=1.5, aspect='equal')

        sizes = (inflow_gdf.upstream_km2 / inflow_gdf.upstream_km2.max() * 300 + 20)
        sc = ax1.scatter(
            inflow_gdf.geometry.x, inflow_gdf.geometry.y,
            s=sizes, c=inflow_gdf.upstream_km2,
            cmap='YlOrRd', zorder=5, edgecolors='k', linewidths=0.5
        )
        plt.colorbar(sc, ax=ax1, label='Upstream area (km²)', shrink=0.8)

        # Label top 3 inflows
        for _, row in inflow_gdf.head(3).iterrows():
            ax1.annotate(f"#{int(row.inflow_rank)}\n{row.upstream_km2:.1f}km²",
                         (row.geometry.x, row.geometry.y),
                         textcoords='offset points', xytext=(6, 6),
                         fontsize=7, color='darkred')

        ax1.set_title("A — Inflow Candidate Locations\n(size/colour = upstream area)")
        ax1.set_xlabel("Easting"); ax1.set_ylabel("Northing")
        ax1.ticklabel_format(style='sci', scilimits=(0,0))
    else:
        ax1.text(0.5, 0.5, "No inflow candidates found",
                 ha='center', va='center', transform=ax1.transAxes)
        ax1.set_title("A — Inflow Candidate Locations")

    # ---- Panel B: Lake water area — S2 summer 2022–2024 + JRC context ----
    if lake_area_ts is not None and len(lake_area_ts) > 0:
        ts = lake_area_ts.sort_values('year')

        # S2 summer water area bars for 2022–2024
        if 's2_summer_water_km2' in ts.columns:
            bars = ax2.bar(ts.year, ts.s2_summer_water_km2,
                           color='steelblue', alpha=0.8, label='S2 summer water area')
            # Add value labels on bars
            for bar, val in zip(bars, ts.s2_summer_water_km2):
                if not np.isnan(val):
                    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

        # JRC long-term ever-water extent as a reference line
        if 'jrc_ever_water_km2' in ts.columns:
            jrc_ref = ts['jrc_ever_water_km2'].dropna()
            if len(jrc_ref) > 0:
                ax2.axhline(jrc_ref.iloc[0], color='navy', linestyle='--',
                            linewidth=1.5, label=f'JRC max ever-water ({jrc_ref.iloc[0]:.3f} km²)')

        ax2.set_xlabel("Year")
        ax2.set_ylabel("Water area (km²)")
        ax2.set_title("""B — Lake Open Water Area
(S2 summer 2022–2024, JRC long-term reference)""")
        ax2.set_xticks(S2_YEARS)
        ax2.legend(fontsize=7)
        ax2.grid(True, alpha=0.3, axis='y')
    else:
        ax2.text(0.5, 0.5, "No lake area data available",
                 ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title("B — Lake Open Water Area")

    # ---- Panel C: Inflow activity heatmap ----
    if inflow_ts is not None and len(inflow_ts) > 0:
        pivot = inflow_ts.pivot_table(
            index='inflow_rank', columns='year',
            values='confidence', aggfunc='mean'
        )
        if len(pivot) > 0:
            im = ax3.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                            vmin=0, vmax=3, interpolation='nearest')
            ax3.set_xticks(range(len(pivot.columns)))
            ax3.set_xticklabels(pivot.columns.astype(int), rotation=45, ha='right', fontsize=7)
            ax3.set_yticks(range(len(pivot.index)))
            ax3.set_yticklabels([f"Inflow #{int(i)}" for i in pivot.index], fontsize=8)
            plt.colorbar(im, ax=ax3, label='Detection confidence (0–3)', shrink=0.8)
            ax3.set_title("C — Inflow Detection Confidence\n(0=none, 3=S2+SAR+inundation agree)")
        else:
            ax3.text(0.5, 0.5, "Insufficient data for heatmap",
                     ha='center', va='center', transform=ax3.transAxes)
            ax3.set_title("C — Inflow Detection Confidence")
    else:
        ax3.text(0.5, 0.5, "No time series data available",
                 ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title("C — Inflow Detection Confidence")

    # ---- Panel D: MNDWI at top 3 inflows ----
    if inflow_ts is not None and len(inflow_ts) > 0:
        top3 = inflow_ts[inflow_ts.inflow_rank <= 3]
        colours = {1: 'red', 2: 'orange', 3: 'gold'}
        for rank in sorted(top3.inflow_rank.unique()):
            sub = top3[top3.inflow_rank == rank].sort_values('year')
            if 'mndwi' in sub.columns:
                ax4.plot(sub.year, sub.mndwi,
                         marker='o', markersize=4, linewidth=1.5,
                         color=colours.get(rank, 'grey'),
                         label=f"Inflow #{int(rank)} "
                               f"({sub.upstream_km2.iloc[0]:.1f} km²)")
        ax4.axhline(MNDWI_WATER_THRESH, color='navy', linestyle='--',
                    linewidth=1, label=f'Water threshold ({MNDWI_WATER_THRESH})')
        ax4.fill_between(
            ax4.get_xlim() if ax4.get_xlim() != (0.0, 1.0) else [min(S2_YEARS), max(S2_YEARS)],
            MNDWI_WATER_THRESH, 1,
            alpha=0.05, color='blue'
        )
        ax4.set_xlabel("Year")
        ax4.set_ylabel("MNDWI (summer)")
        ax4.set_title("D — MNDWI at Top 3 Inflow Points\n(above dashed line = water detected)")
        ax4.set_xticks(S2_YEARS)
        ax4.legend(fontsize=7)
        ax4.grid(True, alpha=0.3)
        ax4.set_ylim(-1, 1)
    else:
        ax4.text(0.5, 0.5, "No MNDWI time series available",
                 ha='center', va='center', transform=ax4.transAxes)
        ax4.set_title("D — MNDWI at Top 3 Inflow Points")

    # Save
    out_path = os.path.join(OUTPUT_DIR, "plots", f"lake_{lake_id}_summary.png")
    os.makedirs(os.path.dirname(out_path), exist_ok=True) # Ensure directory exists
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"    Plot saved: {out_path}")

# Main - run analysis for all lakes

In [ ]:
def run_all_lakes():
    print("\n" + "="*60)
    print("LAKE INFLOW ANALYSIS — START")
    print("="*60)

    print(f"\nLoading {len(LAKE_FILES)} lake shapefiles...")

    # Master results collectors
    all_inflow_points = []
    all_catchments    = []
    master_records    = []

    for lake_id, shp_path in LAKE_FILES.items():
        if not os.path.exists(shp_path):
            print(f"\n  WARNING: File not found for {lake_id} — {shp_path} — skipping.")
            continue

        lake_gdf = gpd.read_file(shp_path)
        print(f"\n  Loaded {lake_id}: {len(lake_gdf)} polygon(s), CRS: {lake_gdf.crs}")

        # Dissolve to single polygon in case shapefile has multiple features
        if len(lake_gdf) > 1:
            print(f"  Dissolving {len(lake_gdf)} features into single lake polygon...")
            lake_gdf = lake_gdf.dissolve().reset_index(drop=True)

        # Ensure projected CRS for accurate buffering and area calculations
        if lake_gdf.crs.is_geographic:
            print(f"  Reprojecting {lake_id} to UTM Zone 6N (EPSG:32606)...")
            lake_gdf = lake_gdf.to_crs(epsg=32606)

        print(f"\n{'='*60}")
        print(f"Processing Lake: {lake_id}")
        print(f"{'='*60}")

        # ---- Stage 1: Flow network ----
        streams_gdf, inflow_gdf, catch_poly, upa_stats = extract_stream_network(
            lake_gdf, lake_id
        )

        # Save inflow points
        if inflow_gdf is not None and len(inflow_gdf) > 0:
            inflow_gdf['lake_id'] = lake_id
            all_inflow_points.append(inflow_gdf)

        # Save catchment polygon
        if catch_poly is not None:
            catch_gdf = gpd.GeoDataFrame(
                [{"lake_id": lake_id, 'geometry': catch_poly,
                  **upa_stats}],
                crs=inflow_gdf.crs if inflow_gdf is not None else lake_gdf.crs
            )
            all_catchments.append(catch_gdf)

        # ---- Stage 2 & 3: Water detection across 2022–2024 ----
        print(f"\n  [STAGE 2+3] Water detection (summer 2022–2024) — Lake {lake_id}")
        inflow_ts    = build_inflow_timeseries(lake_gdf, inflow_gdf, lake_id)
        lake_area_ts = build_lake_area_timeseries(lake_gdf, lake_id)

        # Save per-lake detection CSV
        if len(inflow_ts) > 0:
            ts_path = os.path.join(OUTPUT_DIR, f"lake_{lake_id}_inflow_detection.csv")
            inflow_ts.to_csv(ts_path, index=False)
            print(f"    Inflow detection saved: {ts_path}")

        if len(lake_area_ts) > 0:
            area_path = os.path.join(OUTPUT_DIR, f"lake_{lake_id}_water_area.csv")
            lake_area_ts.to_csv(area_path, index=False)
            print(f"    Lake water area saved: {area_path}")

        # ---- Stage 4: Plot ----
        print(f"\n  [STAGE 4] Plotting — Lake {lake_id}")
        plot_lake_summary(lake_id, inflow_gdf, inflow_ts, lake_area_ts,
                          upa_stats, lake_gdf)

        # ---- Master summary row ----
        master_row = {
            'lake_id':             lake_id,
            'lake_area_km2':       lake_gdf.geometry.area.iloc[0] / 1e6,
            'n_inflow_candidates': upa_stats.get('n_inflows', 0),
            'max_upstream_km2':    upa_stats.get('max_upa_km2', np.nan),
            'total_upstream_km2':  upa_stats.get('total_upa_km2', np.nan),
        }

        # Inflow detection summary across 2022–2024
        if len(inflow_ts) > 0 and 'active_inflow' in inflow_ts.columns:
            master_row['n_seasons_any_inflow'] = (
                inflow_ts.groupby('year')['active_inflow'].any().sum()
            )
            master_row['mean_confidence_2022_2024'] = float(inflow_ts['confidence'].mean())
            for yr in S2_YEARS:
                yr_data = inflow_ts[inflow_ts.year == yr]
                master_row[f'max_confidence_{yr}'] = (
                    float(yr_data['confidence'].max()) if len(yr_data) > 0 else np.nan
                )

        # Lake water area from S2 summer composites
        if len(lake_area_ts) > 0 and 's2_summer_water_km2' in lake_area_ts.columns:
            for yr in S2_YEARS:
                yr_data = lake_area_ts[lake_area_ts.year == yr]
                master_row[f's2_water_km2_{yr}'] = (
                    float(yr_data['s2_summer_water_km2'].iloc[0])
                    if len(yr_data) > 0 else np.nan
                )
        if len(lake_area_ts) > 0 and 'jrc_ever_water_km2' in lake_area_ts.columns:
            master_row['jrc_ever_water_km2'] = float(
                lake_area_ts['jrc_ever_water_km2'].dropna().iloc[0]
                if len(lake_area_ts['jrc_ever_water_km2'].dropna()) > 0 else np.nan
            )

        master_records.append(master_row)

    # ============================================================
    # SAVE MASTER OUTPUTS
    # ============================================================

    print(f"\n{'='*60}")
    print("Saving master outputs...")

    # Master CSV
    master_df = pd.DataFrame(master_records)
    master_path = os.path.join(OUTPUT_DIR, "master_lake_inflow_summary.csv")
    master_df.to_csv(master_path, index=False)
    print(f"  Master summary CSV: {master_path}")
    print(master_df.to_string(index=False))

    # All inflow points GeoPackage
    if all_inflow_points:
        all_pts = pd.concat(all_inflow_points, ignore_index=True)
        final_pts_path = os.path.join(OUTPUT_DIR, "vectors", "all_inflow_points.gpkg")
        temp_pts_path = os.path.join("/tmp", "all_inflow_points.gpkg") # Write to local /tmp first
        try:
            all_pts.to_file(temp_pts_path, driver='GPKG')
            shutil.copy(temp_pts_path, final_pts_path)
            print(f"  Inflow points GeoPackage: {final_pts_path}")
        except Exception as e:
            print(f"  ERROR saving inflow points GeoPackage: {e}")

    # All catchments GeoPackage
    if all_catchments:
        all_catch = pd.concat(all_catchments, ignore_index=True)
        final_catch_path = os.path.join(OUTPUT_DIR, "vectors", "all_catchments.gpkg")
        temp_catch_path = os.path.join("/tmp", "all_catchments.gpkg") # Write to local /tmp first
        try:
            all_catch.to_file(temp_catch_path, driver='GPKG')
            shutil.copy(temp_catch_path, final_catch_path)
            print(f"  Catchments GeoPackage: {final_catch_path}")
        except Exception as e:
            print(f"  ERROR saving catchments GeoPackage: {e}")

    print(f"\n{'='*60}")
    print("ANALYSIS COMPLETE")
    print(f"All outputs saved to: {OUTPUT_DIR}")
    print("="*60)

    return master_df

In [ ]:
if __name__ == "__main__":
    master = run_all_lakes()


LAKE INFLOW ANALYSIS — START

Loading 2 lake shapefiles...

  Loaded lake_1: 1 polygon(s), CRS: EPSG:4326
  Reprojecting lake_1 to UTM Zone 6N (EPSG:32606)...

Processing Lake: lake_1

  [STAGE 1] Flow network — Lake lake_1
    Stream pixels in buffer zone: 2136064
    Inflow candidate pixels: 5
    Inflow candidates: 5
    Largest upstream area: 2.13 km²

  [STAGE 2+3] Water detection (summer 2022–2024) — Lake lake_1


    Lake water area lake_1:  67%|██████▋   | 2/3 [00:02<00:01,  1.06s/it]

    Lake water area lake_1: 100%|██████████| 3/3 [00:02<00:00,  1.07it/s]

    Inflow detection saved: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/lake_lake_1_inflow_detection.csv
    Lake water area saved: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/lake_lake_1_water_area.csv

  [STAGE 4] Plotting — Lake lake_1


    Plot saved: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/plots/lake_lake_1_summary.png

  Loaded lake_2: 1 polygon(s), CRS: EPSG:4326
  Reprojecting lake_2 to UTM Zone 6N (EPSG:32606)...

Processing Lake: lake_2

  [STAGE 1] Flow network — Lake lake_2
    Stream pixels in buffer zone: 2136064
    Inflow candidate pixels: 4
    Inflow candidates: 4
    Largest upstream area: 3.55 km²

  [STAGE 2+3] Water detection (summer 2022–2024) — Lake lake_2


    Lake water area lake_2:   0%|          | 0/3 [00:00<?, ?it/s]

    Lake water area lake_2:  67%|██████▋   | 2/3 [00:00<00:00,  5.75it/s]

    Lake water area lake_2: 100%|██████████| 3/3 [00:00<00:00,  4.61it/s]

    Inflow detection saved: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/lake_lake_2_inflow_detection.csv
    Lake water area saved: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/lake_lake_2_water_area.csv

  [STAGE 4] Plotting — Lake lake_2


    Plot saved: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/plots/lake_lake_2_summary.png

Saving master outputs...
  Master summary CSV: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/master_lake_inflow_summary.csv
lake_id  lake_area_km2  n_inflow_candidates  max_upstream_km2  total_upstream_km2  n_seasons_any_inflow  mean_confidence_2022_2024  max_confidence_2022  max_confidence_2023  max_confidence_2024  jrc_ever_water_km2
 lake_1       0.359848                    5          2.126503            7.612917                     3                   0.266667                  2.0                  1.0                  1.0        6.245951e-11
 lake_2       0.321043                    4          3.545259            8.141228                     1                   0.083333                  1.0                  0.0                  0.0        5.258219e-11
  Inflow points GeoPackage: /content/drive/MyDrive/MRes/Alaska_TRV_Inflow_analysis/Outputs/vectors/all_infl